# Aspect Informativeness

In [ ]:
# %% [markdown]
# # Aspect Informativeness Analysis for Narrative Similarity
#
# Evaluates three aspect extraction versions and a merged optimal cache.
# For each version × aspect combination we compute:
# - **% pos > neg**: fraction of triples where positive similarity > negative similarity
# - **mean_diff**: average (pos_sim − neg_sim)
# - **p-value**: paired t-test (H₀: no difference between pos and neg similarities)
#
# A p-value < 0.05 and % pos>neg clearly above 50% indicates the aspect
# is statistically informative for narrative similarity.

# %% [code]
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR   = "/kaggle/input/datasets/anisoaraana/aspect-aware-narrative-similarity/"
DEV_PATH   = DATA_DIR + "dev_track_a.jsonl"

# Three extraction versions + merged cache
CACHE_PATHS = {
    "V1 (narrative prose)": DATA_DIR + "aspects_cache_v1.json",
    "V2 (role-label steps)": DATA_DIR + "aspects_cache_v2.json",
    "V3 (compact phrases)": DATA_DIR + "aspects_cache_v3.json",
    "V4 (merged optimal)":   DATA_DIR + "merged_aspects_cache.json",
}

MODEL_NAME  = "sentence-transformers/all-roberta-large-v1"
MAX_LEN     = 384
BATCH_SIZE  = 16
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
VIEWS       = ["full", "coa", "outcomes", "theme"]

print(f"Device: {DEVICE}")

# ── Load model ────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print("Model loaded:", MODEL_NAME)


# ── Helpers ───────────────────────────────────────────────────────────────────
def _norm(text):
    return " ".join(str(text).split())


def load_cache(path):
    if not Path(path).exists():
        print(f"  Cache not found: {path}")
        return {}
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    cache = {}
    if isinstance(raw, dict):
        for k, v in raw.items():
            cache[_norm(k)] = {
                "coa":      v.get("coa") or v.get("course_of_action") or "",
                "outcomes": v.get("outcomes") or v.get("outcome") or "",
                "theme":    v.get("theme") or v.get("abstract_theme") or "",
            }
    print(f"  Loaded {len(cache)} entries")
    return cache


def get_view_text(story_text, view, cache):
    if view == "full":
        return story_text
    entry = cache.get(_norm(story_text), {})
    val   = entry.get(view, "")
    return val if val.strip() else story_text   # fallback to full text


def encode_texts(texts):
    embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch   = texts[i:i+BATCH_SIZE]
        encoded = tokenizer(batch, padding=True, truncation=True,
                            max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out   = model(**encoded)
            tok   = out.last_hidden_state
            mask  = encoded["attention_mask"].unsqueeze(-1).float()
            emb   = (tok * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            emb   = F.normalize(emb, p=2, dim=1)
        embeddings.append(emb.cpu().numpy())
    return np.vstack(embeddings)


def evaluate_view(triples, view, cache):
    """Compute paired pos/neg similarities for one view."""
    unique = {}
    for anchor, ta, tb, _ in triples:
        for t in (anchor, ta, tb):
            if t not in unique:
                unique[t] = get_view_text(t, view, cache)

    texts  = list(unique.keys())
    embs   = encode_texts([unique[t] for t in texts])
    t2emb  = {t: torch.tensor(embs[i], device=DEVICE) for i, t in enumerate(texts)}

    pos_sims, neg_sims = [], []
    for anchor, ta, tb, closer in triples:
        ea   = t2emb[anchor].unsqueeze(0)
        epos = t2emb[ta if closer else tb].unsqueeze(0)
        eneg = t2emb[tb if closer else ta].unsqueeze(0)
        pos_sims.append(F.cosine_similarity(ea, epos).item())
        neg_sims.append(F.cosine_similarity(ea, eneg).item())

    pos_arr = np.array(pos_sims)
    neg_arr = np.array(neg_sims)
    diff    = pos_arr - neg_arr
    t_stat, p_val = stats.ttest_rel(pos_arr, neg_arr)

    # Cache hit rate (% of stories where aspect text differs from full text)
    hit_rate = sum(
        1 for t in texts
        if get_view_text(t, view, cache) != t
    ) / max(len(texts), 1) * 100

    return {
        "mean_pos":   float(pos_arr.mean()),
        "mean_neg":   float(neg_arr.mean()),
        "mean_diff":  float(diff.mean()),
        "pct_pos_gt_neg": float((diff > 0).mean() * 100),
        "t_stat":     float(t_stat),
        "p_value":    float(p_val),
        "cache_hit_%": hit_rate,
        "pos_sims":   pos_sims,
        "neg_sims":   neg_sims,
    }


# ── Load data ─────────────────────────────────────────────────────────────────
def load_triples(path):
    triples = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            triples.append((
                obj["anchor_text"], obj["text_a"], obj["text_b"],
                obj["text_a_is_closer"]
            ))
    return triples

triples = load_triples(DEV_PATH)
print(f"\nLoaded {len(triples)} dev triples")


# %% [markdown]
# ## Run Evaluation Across All Cache Versions

# %% [code]
all_results  = {}   # version → view → metrics dict
raw_sims     = {}   # version → view → {"pos": [...], "neg": [...]}

for version_name, cache_path in CACHE_PATHS.items():
    print(f"\n{'='*55}")
    print(f"  {version_name}")
    print(f"{'='*55}")
    cache = load_cache(cache_path)
    all_results[version_name] = {}
    raw_sims[version_name]    = {}

    for view in VIEWS:
        print(f"  Evaluating {view} …", end=" ", flush=True)
        res = evaluate_view(triples, view, cache)
        all_results[version_name][view] = res
        raw_sims[version_name][view]    = {
            "pos": res.pop("pos_sims"),
            "neg": res.pop("neg_sims"),
        }
        sig = "✓ p<0.05" if res["p_value"] < 0.05 else "✗"
        print(f"  pct={res['pct_pos_gt_neg']:.1f}%  "
              f"diff={res['mean_diff']:.4f}  "
              f"p={res['p_value']:.4f}  {sig}")


# %% [markdown]
# ## Summary Table

# %% [code]
rows = []
for vname, views_dict in all_results.items():
    for view, metrics in views_dict.items():
        rows.append({
            "Version":      vname,
            "Aspect":       view,
            "% pos>neg":    round(metrics["pct_pos_gt_neg"], 1),
            "mean_diff":    round(metrics["mean_diff"], 4),
            "mean_pos_sim": round(metrics["mean_pos"], 4),
            "mean_neg_sim": round(metrics["mean_neg"], 4),
            "p-value":      round(metrics["p_value"], 4),
            "Significant":  "✓" if metrics["p_value"] < 0.05 else "",
            "Cache hit %":  round(metrics["cache_hit_%"], 1),
        })

df = pd.DataFrame(rows)
# Pivot for readability
pivot_pct = df.pivot(index="Version", columns="Aspect", values="% pos>neg")[VIEWS]
pivot_p   = df.pivot(index="Version", columns="Aspect", values="p-value")[VIEWS]
pivot_diff= df.pivot(index="Version", columns="Aspect", values="mean_diff")[VIEWS]

print("\n── % pos > neg (higher is better, random baseline = 50%) ──")
print(pivot_pct.to_string())

print("\n── p-value (< 0.05 = statistically significant) ──")
print(pivot_p.to_string())

print("\n── mean_diff = mean(pos_sim − neg_sim) (higher = cleaner separation) ──")
print(pivot_diff.to_string())

# Full table
print("\n── Full table ──")
print(df.to_string(index=False))

df.to_csv("aspect_informativeness_all_versions.csv", index=False)
print("\nSaved: aspect_informativeness_all_versions.csv")


# %% [markdown]
# ## Visualisation 1 — % pos > neg heatmap across versions and aspects

# %% [code]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Aspect Informativeness: % positive > negative similarity",
             fontsize=14, fontweight="bold", y=1.02)

titles = ["% pos > neg (higher = better)",
          "mean_diff = pos−neg sim (higher = better)",
          "−log10(p-value) (higher = more significant)"]

data_matrices = [
    pivot_pct,
    pivot_diff,
    pivot_p.applymap(lambda p: -np.log10(max(p, 1e-10))),
]
cmaps   = ["RdYlGn", "RdYlGn", "RdYlGn"]
centers = [50,        0,         -np.log10(0.05)]
fmts    = [".1f",     ".4f",      ".2f"]

for ax, data, title, cmap, center, fmt in zip(
        axes, data_matrices, titles, cmaps, centers, fmts):
    sns.heatmap(
        data, ax=ax, annot=True, fmt=fmt, cmap=cmap,
        center=center, linewidths=0.5,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Aspect")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig("aspect_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: aspect_heatmap.png")


# %% [markdown]
# ## Visualisation 2 — Best version per aspect: similarity distributions

# %% [code]
# Select the best version per aspect based on % pos > neg
best_by_aspect = {}
for view in ["coa", "outcomes", "theme"]:
    best_v, best_pct = None, -1
    for vname, vdata in all_results.items():
        if vname == "V4 (merged optimal)":
            continue   # skip merged for this comparison
        pct = vdata[view]["pct_pos_gt_neg"]
        if pct > best_pct:
            best_pct = pct
            best_v   = vname
    best_by_aspect[view] = (best_v, best_pct)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Similarity distributions — best extraction version per aspect",
             fontsize=13, fontweight="bold")

for ax, view in zip(axes, ["coa", "outcomes", "theme"]):
    best_v, best_pct = best_by_aspect[view]
    pos = raw_sims[best_v][view]["pos"]
    neg = raw_sims[best_v][view]["neg"]
    p   = all_results[best_v][view]["p_value"]
    sig = "p={:.4f} ✓".format(p) if p < 0.05 else "p={:.4f} ✗".format(p)

    ax.hist(pos, bins=25, alpha=0.55, color="#2ecc71", label="Positive")
    ax.hist(neg, bins=25, alpha=0.55, color="#e74c3c", label="Negative")
    ax.axvline(np.mean(pos), color="#27ae60", linestyle="--", linewidth=1.5)
    ax.axvline(np.mean(neg), color="#c0392b", linestyle="--", linewidth=1.5)
    ax.set_title(f"{view.upper()} — {best_v}\n"
                 f"{best_pct:.1f}% pos>neg   {sig}", fontsize=9)
    ax.set_xlabel("Cosine similarity")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("aspect_distributions_best.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: aspect_distributions_best.png")


# %% [markdown]
# ## Visualisation 3 — Full text vs best aspect per version (bar chart)

# %% [code]
fig, ax = plt.subplots(figsize=(12, 5))

versions  = list(all_results.keys())
x         = np.arange(len(versions))
bar_w     = 0.18
colors    = {"full": "#95a5a6", "coa": "#3498db",
             "outcomes": "#e67e22", "theme": "#9b59b6"}

for i, view in enumerate(VIEWS):
    pcts = [all_results[v][view]["pct_pos_gt_neg"] for v in versions]
    bars = ax.bar(x + i * bar_w, pcts, bar_w, label=view,
                  color=colors[view], alpha=0.85, edgecolor="white")
    # Mark significant bars
    for bar, vname in zip(bars, versions):
        p = all_results[vname][view]["p_value"]
        if p < 0.05:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.3, "✓",
                    ha="center", va="bottom", fontsize=9, color="black")

ax.axhline(50, color="black", linestyle="--", linewidth=1, label="Random (50%)")
ax.set_xticks(x + bar_w * (len(VIEWS)-1) / 2)
ax.set_xticklabels([v.split("(")[0].strip() for v in versions], rotation=15)
ax.set_ylabel("% positive > negative similarity")
ax.set_title("Aspect Informativeness by Extraction Version\n"
             "(✓ = p < 0.05, dashed line = random baseline)",
             fontsize=11)
ax.set_ylim(44, 68)
ax.legend(title="Aspect", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("aspect_informativeness_bars.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: aspect_informativeness_bars.png")


# %% [code]
print("=" * 70)
print("  INTERPRETATION")
print("=" * 70)

sig_threshold = 0.05
for view in VIEWS:
    print(f"\n── {view.upper()} ──")
    sig_versions = [
        (vname, all_results[vname][view])
        for vname in all_results
        if all_results[vname][view]["p_value"] < sig_threshold
    ]
    if not sig_versions:
        print(f"  No version is statistically significant (p < {sig_threshold})")
        print(f"  → {view} does NOT reliably discriminate narrative similarity")
        print(f"    in this dataset. Avoid as the sole input for training.")
    else:
        for vname, metrics in sorted(sig_versions,
                                     key=lambda x: -x[1]["pct_pos_gt_neg"]):
            print(f"  {vname}: {metrics['pct_pos_gt_neg']:.1f}% pos>neg, "
                  f"p={metrics['p_value']:.4f}")
        best_vname = max(sig_versions, key=lambda x: x[1]["pct_pos_gt_neg"])[0]
        print(f"  → Best version: {best_vname}")

# Save final recommendation table
rec = []
for view in VIEWS:
    all_v = [(vname, all_results[vname][view]) for vname in all_results
             if vname != "V4 (merged optimal)"]
    best  = max(all_v, key=lambda x: x[1]["pct_pos_gt_neg"])
    sig   = best[1]["p_value"] < 0.05
    rec.append({
        "Aspect":      view,
        "Best version": best[0],
        "% pos>neg":   round(best[1]["pct_pos_gt_neg"], 1),
        "p-value":     round(best[1]["p_value"], 4),
        "Informative": "Yes ✓" if sig else "No ✗",
    })

df_rec = pd.DataFrame(rec)
print(df_rec.to_string(index=False))
df_rec.to_csv("aspect_recommendations.csv", index=False)
print("\nSaved: aspect_recommendations.csv")